# Controlador Proporcional e Integral

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------------------------------------
# Parámetros
# -------------------------------------------------------

kp = 1.4                       # Ganancia proporcional fija
t = np.linspace(0, 20, 600)    # Vector de tiempo


# -------------------------------------------------------
# Respuesta con control P para comparación
# -------------------------------------------------------

system_P = signal.TransferFunction(
    [kp],
    [1, 1.4, 1 + kp]
)

t_P, y_P = signal.step(system_P, T=t)

# Valor final del sistema con control P
y_final_P = kp / (1 + kp)


# -------------------------------------------------------
# Slider para ki
# -------------------------------------------------------

ki_slider = widgets.FloatSlider(
    value=0.2,
    min=0.05,
    max=4.0,
    step=0.05,
    description='ki:',
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()


# -------------------------------------------------------
# Función de simulación
# -------------------------------------------------------

def simulate(change=None):

    ki = ki_slider.value

    # ---------------------------------------------------
    # Sistema PI en lazo cerrado
    #
    #             kp*s + ki
    # T(s) = --------------------------
    #        s³ + 1.4s² + (1+kp)s + ki
    # ---------------------------------------------------

    system_PI = signal.TransferFunction(
        [kp, ki],
        [1, 1.4, 1 + kp, ki]
    )

    # Respuesta al escalón
    tout, y = signal.step(system_PI, T=t)

    # Polos
    poles = np.roots(
        [1, 1.4, 1 + kp, ki]
    )

    # Verificar estabilidad
    stable = np.all(np.real(poles) < 0)


    # ---------------------------------------------------
    # Indicadores
    # ---------------------------------------------------

    if stable:

        # Para un PI estable y ki > 0:
        # y(infinito) = 1
        y_final = 1.0
        ess = 0.0

        # Sobrepaso respecto a la referencia
        y_max = np.max(y)

        overshoot = max(
            0,
            (y_max - y_final) / y_final * 100
        )

        # Tiempo de establecimiento al 2 %
        tolerance = 0.02 * y_final

        outside = np.where(
            np.abs(y - y_final) > tolerance
        )[0]

        if len(outside) == 0:
            ts = 0.0
        elif outside[-1] < len(t) - 1:
            ts = t[outside[-1] + 1]
        else:
            ts = np.nan

    else:

        y_final = np.nan
        ess = np.nan
        overshoot = np.nan
        ts = np.nan


    # ---------------------------------------------------
    # Mostrar resultados
    # ---------------------------------------------------

    with output:

        clear_output(wait=True)

        fig, ax = plt.subplots(1, 2, figsize=(11, 4))


        # =================================================
        # GRÁFICA 1: respuesta temporal
        # =================================================

        ax[0].plot(
            t_P,
            y_P,
            linestyle='--',
            label=f'Control P, kp = {kp}'
        )

        ax[0].plot(
            tout,
            y,
            linewidth=2,
            label=f'Control PI, ki = {ki:.2f}'
        )

        ax[0].axhline(
            1,
            linestyle=':',
            label='Referencia'
        )

        ax[0].set_xlabel('Tiempo [s]')
        ax[0].set_ylabel('Salida')
        ax[0].set_title('Respuesta al escalón')
        ax[0].grid()
        ax[0].legend()


        # =================================================
        # GRÁFICA 2: polos
        # =================================================

        ax[1].scatter(
            poles.real,
            poles.imag,
            marker='x',
            s=100
        )

        ax[1].axhline(0, linewidth=1)
        ax[1].axvline(0, linewidth=1)

        ax[1].set_xlim(-2, 1)
        ax[1].set_ylim(-3, 3)

        ax[1].set_xlabel('Parte real')
        ax[1].set_ylabel('Parte imaginaria')
        ax[1].set_title('Polos del sistema PI')
        ax[1].grid()

        plt.tight_layout()
        plt.show()


        # =================================================
        # RESULTADOS
        # =================================================

        print(f"kp = {kp:.2f}")
        print(f"ki = {ki:.2f}")
        print()

        if stable:

            print("Sistema estable")
            print(f"Valor final y(∞)          = {y_final:.3f}")
            print(f"Error estacionario e_ss   = {ess:.3f}")
            print(f"Sobrepaso                 = {overshoot:.1f} %")

            if np.isnan(ts):
                print("T. establecimiento        > 20 s")
            else:
                print(f"T. establecimiento        = {ts:.2f} s")

        else:

            print("Sistema INESTABLE")
            print("El valor final, el error estacionario y")
            print("el tiempo de establecimiento no están definidos.")

        print("\nPolos:")

        for p in poles:
            print(f"   {p:.4f}")


# -------------------------------------------------------
# Actualizar cuando cambia ki
# -------------------------------------------------------

ki_slider.observe(simulate, names='value')

display(ki_slider)
display(output)

simulate()

FloatSlider(value=0.2, continuous_update=False, description='ki:', layout=Layout(width='500px'), max=4.0, min=…

Output()